# Final Results Figures

This notebook is the final thesis-summary synthesis step. It reads only completed result CSVs, creates mutually exclusive tokenization-fragmentation categories, and writes thesis-ready `0801-*` summary tables and figures.

It does not run transformer inference, embedding extraction, context generation, or any upstream experiment code.

## Imports and paths

The notebook assumes it is run from the project root. All outputs use the `0801` prefix so earlier `010x`-`070x` outputs are not overwritten.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

try:
    from IPython.display import Image, display
except ImportError:
    Image = None

    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().resolve()
RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for directory in [RESULTS_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = {
    "isolated_pairs": RESULTS_DIR / "0303-isolated_all_models_pair_similarities_with_tokenization.csv",
    "contextual_pairs": RESULTS_DIR / "0503-contextual_all_models_pair_similarities_with_tokenization.csv",
    "combined_summary": RESULTS_DIR / "0504-isolated_vs_contextual_summary.csv",
    "triplet_robustness": RESULTS_DIR / "0604-triplet_probe_robustness.csv",
    "triplet_thesis": TABLE_DIR / "0706-triplet_probe_thesis_summary.csv",
}

OUTPUT_FILES = {
    "best_overall_configurations": TABLE_DIR / "0801-best_overall_configurations.csv",
    "best_isolated_vs_contextual_configurations": TABLE_DIR / "0801-best_isolated_vs_contextual_configurations.csv",
    "clean_one_both_spearman_summary": TABLE_DIR / "0801-clean_one_both_spearman_summary.csv",
    "fragmentation_category_strategy_summary": TABLE_DIR / "0801-fragmentation_category_strategy_summary.csv",
    "top3_spearman_by_fragmentation_model_condition": TABLE_DIR / "0801-top3_spearman_by_fragmentation_model_condition.csv",
    "contextual_delta_by_configuration": TABLE_DIR / "0801-contextual_minus_isolated_delta_by_configuration.csv",
    "triplet_probe_summary_with_caveats": TABLE_DIR / "0801-triplet_probe_summary_with_caveats.csv",
    "best_spearman_by_model_condition_figure": FIGURE_DIR / "0801-best_spearman_by_model_condition.png",
    "spearman_by_fragmentation_category_figure": FIGURE_DIR / "0801-spearman_by_fragmentation_category.png",
    "contextual_delta_by_configuration_figure": FIGURE_DIR / "0801-contextual_minus_isolated_delta_by_configuration.png",
    "triplet_same_root_vs_same_suffix_figure": FIGURE_DIR / "0801-triplet_same_root_vs_same_suffix_summary.png",
}

INPUT_ORDER = ["isolated", "contextual"]
MODEL_ORDER = ["BERTurk", "mBERT", "XLM-R"]
LAYER_ORDER = [1, 7, 12]
POOLING_ORDER = ["first", "last", "mean", "max"]
FRAGMENTATION_ORDER = ["clean", "one_word_split", "both_words_split"]
FRAGMENTATION_LABELS = {
    "clean": "Clean",
    "one_word_split": "One word split",
    "both_words_split": "Both words split",
}

## Load and validate inputs

These checks make the notebook fail early if a required upstream output is missing or if the isolated/contextual pair grids no longer match.

In [ ]:
missing_files = [path for path in REQUIRED_FILES.values() if not path.exists()]
if missing_files:
    missing_text = "\n".join(str(path.relative_to(PROJECT_ROOT)) for path in missing_files)
    raise FileNotFoundError(f"Missing required final-results inputs:\n{missing_text}")

isolated_pairs = pd.read_csv(REQUIRED_FILES["isolated_pairs"])
contextual_pairs = pd.read_csv(REQUIRED_FILES["contextual_pairs"])
combined_summary = pd.read_csv(REQUIRED_FILES["combined_summary"])
triplet_robustness = pd.read_csv(REQUIRED_FILES["triplet_robustness"])
triplet_thesis = pd.read_csv(REQUIRED_FILES["triplet_thesis"])

expected_pair_columns = {
    "QID",
    "W1",
    "W2",
    "Sim",
    "model",
    "model_name",
    "layer",
    "pooling",
    "cosine_similarity",
    "word1_is_split",
    "word2_is_split",
    "clean_pair",
    "split_pair",
    "both_words_split",
}
for label, frame in [("isolated_pairs", isolated_pairs), ("contextual_pairs", contextual_pairs)]:
    missing = expected_pair_columns.difference(frame.columns)
    if missing:
        raise ValueError(f"{label} is missing expected columns: {sorted(missing)}")

required_summary_columns = {
    "input_condition",
    "model",
    "model_name",
    "layer",
    "pooling",
    "spearman_rho",
    "p_value",
    "n_pairs",
}
missing_summary = required_summary_columns.difference(combined_summary.columns)
if missing_summary:
    raise ValueError(f"combined_summary is missing expected columns: {sorted(missing_summary)}")

print("Loaded input shapes:")
for label, frame in [
    ("isolated_pairs", isolated_pairs),
    ("contextual_pairs", contextual_pairs),
    ("combined_summary", combined_summary),
    ("triplet_robustness", triplet_robustness),
    ("triplet_thesis", triplet_thesis),
]:
    print(f"  {label:24s} {frame.shape[0]:5d} rows x {frame.shape[1]:2d} columns")

## Fragmentation categories

The category is mutually exclusive and model-specific because it is built from the enriched result files' existing tokenization flags:

- `clean`: neither word split
- `one_word_split`: exactly one word split
- `both_words_split`: both words split

In [ ]:
def bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


def add_input_and_fragmentation(frame: pd.DataFrame, input_condition: str) -> pd.DataFrame:
    out = frame.copy()
    out["input_condition"] = input_condition
    out["layer"] = out["layer"].astype(int)

    word1_split = bool_series(out["word1_is_split"])
    word2_split = bool_series(out["word2_is_split"])
    split_count = word1_split.astype(int) + word2_split.astype(int)

    out["word1_is_split"] = word1_split
    out["word2_is_split"] = word2_split
    out["fragmentation_category"] = np.select(
        [split_count.eq(0), split_count.eq(1), split_count.eq(2)],
        ["clean", "one_word_split", "both_words_split"],
        default="unknown",
    )
    out["fragmentation_category"] = pd.Categorical(
        out["fragmentation_category"],
        categories=FRAGMENTATION_ORDER,
        ordered=True,
    )

    clean_pair = bool_series(out["clean_pair"])
    split_pair = bool_series(out["split_pair"])
    both_split = bool_series(out["both_words_split"])
    if not clean_pair.eq(split_count.eq(0)).all():
        raise AssertionError(f"{input_condition}: clean_pair does not match word-level split flags")
    if not split_pair.eq(split_count.gt(0)).all():
        raise AssertionError(f"{input_condition}: split_pair does not match word-level split flags")
    if not both_split.eq(split_count.eq(2)).all():
        raise AssertionError(f"{input_condition}: both_words_split does not match word-level split flags")

    return out


isolated_pairs_clean = add_input_and_fragmentation(isolated_pairs, "isolated")
contextual_pairs_clean = add_input_and_fragmentation(contextual_pairs, "contextual")

fixed_key = ["QID", "W1", "W2", "model", "layer", "pooling"]
iso_fixed = isolated_pairs_clean[fixed_key + ["Sim", "fragmentation_category"]].copy()
ctx_fixed = contextual_pairs_clean[fixed_key + ["Sim", "fragmentation_category"]].copy()
fixed_check = iso_fixed.merge(
    ctx_fixed,
    on=fixed_key,
    how="outer",
    suffixes=("_isolated", "_contextual"),
    indicator=True,
    validate="one_to_one",
)

if not fixed_check["_merge"].eq("both").all():
    counts = fixed_check["_merge"].value_counts().to_dict()
    raise AssertionError(f"Isolated/contextual pair groups are not fixed: {counts}")
if not np.allclose(fixed_check["Sim_isolated"], fixed_check["Sim_contextual"], equal_nan=True):
    raise AssertionError("Human similarity scores differ between isolated and contextual pair files")
if not fixed_check["fragmentation_category_isolated"].eq(
    fixed_check["fragmentation_category_contextual"]
).all():
    raise AssertionError("Fragmentation categories differ between isolated and contextual pair files")

pair_data = pd.concat([isolated_pairs_clean, contextual_pairs_clean], ignore_index=True)
pair_data["input_condition"] = pd.Categorical(pair_data["input_condition"], INPUT_ORDER, ordered=True)
pair_data["model"] = pd.Categorical(pair_data["model"], MODEL_ORDER, ordered=True)
pair_data["pooling"] = pd.Categorical(pair_data["pooling"], POOLING_ORDER, ordered=True)

fragmentation_counts = (
    pair_data.drop_duplicates(["input_condition", "model", "QID", "fragmentation_category"])
    .groupby(["input_condition", "model", "fragmentation_category"], observed=True)
    .agg(n_pairs=("QID", "nunique"))
    .reset_index()
)
fragmentation_counts["category_share"] = fragmentation_counts["n_pairs"] / 500
fragmentation_counts

## Best overall and condition-specific configurations

These tables use the existing `0504` summary and do not recompute embeddings. The first table is the top overall configuration list; the second gives the best configuration for each model in each input condition.

In [ ]:
def apply_display_order(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if "input_condition" in out.columns:
        out["input_condition"] = pd.Categorical(out["input_condition"], INPUT_ORDER, ordered=True)
    if "model" in out.columns:
        out["model"] = pd.Categorical(out["model"], MODEL_ORDER, ordered=True)
    if "pooling" in out.columns:
        out["pooling"] = pd.Categorical(out["pooling"], POOLING_ORDER, ordered=True)
    if "fragmentation_category" in out.columns:
        out["fragmentation_category"] = pd.Categorical(
            out["fragmentation_category"], FRAGMENTATION_ORDER, ordered=True
        )
    return out


combined = combined_summary.copy()
combined["layer"] = combined["layer"].astype(int)
combined = apply_display_order(combined)

ranked_combined = combined.sort_values("spearman_rho", ascending=False).reset_index(drop=True)
ranked_combined.insert(0, "rank_overall", np.arange(1, len(ranked_combined) + 1))

best_overall_configurations = ranked_combined.head(12)[
    [
        "rank_overall",
        "input_condition",
        "model",
        "model_name",
        "layer",
        "pooling",
        "spearman_rho",
        "p_value",
        "n_pairs",
    ]
].copy()

condition_rank = combined.copy()
condition_rank["rank_within_condition"] = (
    condition_rank.groupby("input_condition", observed=True)["spearman_rho"]
    .rank(method="first", ascending=False)
    .astype(int)
)

best_isolated_vs_contextual_configurations = (
    condition_rank.sort_values("spearman_rho", ascending=False)
    .groupby(["input_condition", "model"], observed=True, as_index=False, sort=False)
    .head(1)
    .copy()
)
best_isolated_vs_contextual_configurations = apply_display_order(
    best_isolated_vs_contextual_configurations
).sort_values(["input_condition", "model"]).reset_index(drop=True)
best_isolated_vs_contextual_configurations = best_isolated_vs_contextual_configurations[
    [
        "input_condition",
        "model",
        "model_name",
        "layer",
        "pooling",
        "spearman_rho",
        "p_value",
        "n_pairs",
        "rank_within_condition",
    ]
]

best_overall_configurations

## Spearman by fragmentation category

Spearman correlations are recomputed from the pair-level cosine similarities within each fixed model-specific fragmentation category. This keeps the same pair groups across isolated and contextual conditions while allowing tokenizer-specific clean/split labels.

In [ ]:
def spearman_summary(group: pd.DataFrame) -> pd.Series:
    usable = group[["Sim", "cosine_similarity"]].dropna()
    n = len(usable)
    if n < 3 or usable["Sim"].nunique() < 2 or usable["cosine_similarity"].nunique() < 2:
        return pd.Series({"spearman_rho": np.nan, "p_value": np.nan, "n_pairs": n})
    rho, p_value = spearmanr(usable["Sim"], usable["cosine_similarity"])
    return pd.Series({"spearman_rho": rho, "p_value": p_value, "n_pairs": n})


fragmentation_category_spearman = (
    pair_data.groupby(
        ["input_condition", "model", "model_name", "layer", "pooling", "fragmentation_category"],
        observed=True,
    )
    .apply(spearman_summary)
    .reset_index()
)

config_totals = (
    pair_data.groupby(["input_condition", "model", "layer", "pooling"], observed=True)
    .agg(total_pairs=("QID", "nunique"))
    .reset_index()
)
fragmentation_category_spearman = fragmentation_category_spearman.merge(
    config_totals,
    on=["input_condition", "model", "layer", "pooling"],
    how="left",
    validate="many_to_one",
)
fragmentation_category_spearman["category_share"] = (
    fragmentation_category_spearman["n_pairs"] / fragmentation_category_spearman["total_pairs"]
)
fragmentation_category_spearman["rank_within_configuration"] = (
    fragmentation_category_spearman.groupby(
        ["input_condition", "model", "layer", "pooling"], observed=True
    )["spearman_rho"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
fragmentation_category_spearman = apply_display_order(fragmentation_category_spearman).sort_values(
    ["input_condition", "model", "fragmentation_category", "layer", "pooling"]
)

best_fragmentation_rows = (
    fragmentation_category_spearman.sort_values("spearman_rho", ascending=False)
    .groupby(["input_condition", "model", "fragmentation_category"], observed=True, as_index=False, sort=False)
    .head(1)
    .rename(
        columns={
            "layer": "best_layer",
            "pooling": "best_pooling",
            "spearman_rho": "max_spearman_rho",
            "p_value": "max_spearman_p_value",
        }
    )
)

fragmentation_category_strategy_summary = (
    fragmentation_category_spearman.groupby(
        ["input_condition", "model", "model_name", "fragmentation_category"], observed=True
    )
    .agg(
        n_pairs=("n_pairs", "first"),
        category_share=("category_share", "first"),
        mean_spearman_rho=("spearman_rho", "mean"),
        median_spearman_rho=("spearman_rho", "median"),
        min_spearman_rho=("spearman_rho", "min"),
        n_significant_p_lt_005=("p_value", lambda values: int((values < 0.05).sum())),
        n_configurations=("spearman_rho", "size"),
    )
    .reset_index()
)
fragmentation_category_strategy_summary = fragmentation_category_strategy_summary.merge(
    best_fragmentation_rows[
        [
            "input_condition",
            "model",
            "fragmentation_category",
            "best_layer",
            "best_pooling",
            "max_spearman_rho",
            "max_spearman_p_value",
        ]
    ],
    on=["input_condition", "model", "fragmentation_category"],
    how="left",
    validate="one_to_one",
)
fragmentation_category_strategy_summary = apply_display_order(
    fragmentation_category_strategy_summary
).sort_values(["fragmentation_category", "input_condition", "model"])

top3_rows = []
for fragmentation_category in FRAGMENTATION_ORDER:
    for model in MODEL_ORDER:
        for input_condition in INPUT_ORDER:
            subset = fragmentation_category_spearman[
                fragmentation_category_spearman["fragmentation_category"].astype(str).eq(fragmentation_category)
                & fragmentation_category_spearman["model"].astype(str).eq(model)
                & fragmentation_category_spearman["input_condition"].astype(str).eq(input_condition)
            ].copy()
            subset = subset.sort_values(["spearman_rho", "p_value"], ascending=[False, True]).head(3)
            row = {
                "pair_type": fragmentation_category,
                "model": model,
                "input_condition": input_condition,
                "n_pairs": int(subset["n_pairs"].iloc[0]) if len(subset) else np.nan,
                "category_share": float(subset["category_share"].iloc[0]) if len(subset) else np.nan,
            }
            for rank in range(1, 4):
                if rank <= len(subset):
                    result_row = subset.iloc[rank - 1]
                    row[f"rank_{rank}_spearman_rho"] = result_row["spearman_rho"]
                    row[f"rank_{rank}_p_value"] = result_row["p_value"]
                    row[f"rank_{rank}_layer"] = int(result_row["layer"])
                    row[f"rank_{rank}_composition_strategy"] = result_row["pooling"]
                else:
                    row[f"rank_{rank}_spearman_rho"] = np.nan
                    row[f"rank_{rank}_p_value"] = np.nan
                    row[f"rank_{rank}_layer"] = np.nan
                    row[f"rank_{rank}_composition_strategy"] = np.nan
            top3_rows.append(row)

top3_spearman_by_fragmentation_model_condition = pd.DataFrame(top3_rows)

fragmentation_category_spearman[
    [
        "input_condition",
        "model",
        "layer",
        "pooling",
        "fragmentation_category",
        "n_pairs",
        "category_share",
        "spearman_rho",
        "p_value",
    ]
].head(12)

## Contextual minus isolated delta

This table keeps model, layer, and pooling fixed, then subtracts isolated Spearman from contextual Spearman.

In [ ]:
isolated_summary = combined.query("input_condition == 'isolated'").copy()
contextual_summary = combined.query("input_condition == 'contextual'").copy()

contextual_delta_by_configuration = isolated_summary.merge(
    contextual_summary,
    on=["model", "model_name", "layer", "pooling"],
    suffixes=("_isolated", "_contextual"),
    validate="one_to_one",
)
contextual_delta_by_configuration = contextual_delta_by_configuration.rename(
    columns={
        "spearman_rho_isolated": "isolated_spearman_rho",
        "spearman_rho_contextual": "contextual_spearman_rho",
        "p_value_isolated": "isolated_p_value",
        "p_value_contextual": "contextual_p_value",
        "n_pairs_isolated": "isolated_n_pairs",
        "n_pairs_contextual": "contextual_n_pairs",
    }
)
contextual_delta_by_configuration["contextual_minus_isolated"] = (
    contextual_delta_by_configuration["contextual_spearman_rho"]
    - contextual_delta_by_configuration["isolated_spearman_rho"]
)
contextual_delta_by_configuration["relative_contextual_gain"] = (
    contextual_delta_by_configuration["contextual_minus_isolated"]
    / contextual_delta_by_configuration["isolated_spearman_rho"].abs().replace(0, np.nan)
)
contextual_delta_by_configuration = contextual_delta_by_configuration.sort_values(
    "contextual_minus_isolated", ascending=False
).reset_index(drop=True)
contextual_delta_by_configuration.insert(
    0, "rank_contextual_gain", np.arange(1, len(contextual_delta_by_configuration) + 1)
)
contextual_delta_by_configuration = contextual_delta_by_configuration[
    [
        "rank_contextual_gain",
        "model",
        "model_name",
        "layer",
        "pooling",
        "isolated_spearman_rho",
        "contextual_spearman_rho",
        "contextual_minus_isolated",
        "relative_contextual_gain",
        "isolated_p_value",
        "contextual_p_value",
        "isolated_n_pairs",
        "contextual_n_pairs",
    ]
]

contextual_delta_by_configuration.head(10)

## Triplet probe caveat table

The triplet probe is kept as exploratory mechanistic evidence. The caveat columns make that status explicit and flag the BERTurk-defined all-split subset.

In [ ]:
triplet_probe_summary_with_caveats = triplet_thesis.copy()
triplet_probe_summary_with_caveats["analysis_role"] = "exploratory_probe"
triplet_probe_summary_with_caveats["main_evidence_for_thesis_claim"] = False
triplet_probe_summary_with_caveats["delta_direction"] = np.where(
    triplet_probe_summary_with_caveats["mean_delta"].ge(0),
    "same_root_closer",
    "same_suffix_closer",
)
triplet_probe_summary_with_caveats["is_berturk_all_split_subset"] = (
    triplet_probe_summary_with_caveats["robustness_subset"].eq("berturk_all_split_triplets")
)
triplet_probe_summary_with_caveats["small_subset_caveat"] = np.where(
    triplet_probe_summary_with_caveats["n"].lt(20),
    "Small subset; interpret as a robustness check, not a standalone estimate.",
    "Not applicable",
)
triplet_probe_summary_with_caveats["subset_definition_caveat"] = np.select(
    [
        triplet_probe_summary_with_caveats["robustness_subset"].eq("berturk_all_split_triplets"),
        triplet_probe_summary_with_caveats["robustness_subset"].eq("model_all_three_split"),
        triplet_probe_summary_with_caveats["robustness_subset"].eq("berturk_shared_final_subtoken"),
    ],
    [
        "Subset is defined by BERTurk tokenization for all models; use only as a BERTurk all-split stress check.",
        "Subset is defined by each model's own tokenization, so sample size is model-dependent.",
        "Subset is defined by BERTurk final-subtoken sharing; it is a suffix-confound stress check.",
    ],
    default="Headline exploratory triplet summary; not main AnlamVer evidence.",
)
triplet_probe_summary_with_caveats["interpretation_caveat"] = (
    "Exploratory triplet probe; interpret after the main AnlamVer Spearman results."
)

triplet_probe_summary_with_caveats = triplet_probe_summary_with_caveats[
    [
        "summary_source",
        "robustness_subset",
        "subset_description",
        "analysis_role",
        "main_evidence_for_thesis_claim",
        "model",
        "model_name",
        "layer",
        "pooling",
        "n",
        "mean_sim_same_root",
        "mean_sim_same_suffix",
        "mean_delta",
        "delta_direction",
        "ci95_low",
        "ci95_high",
        "wilcoxon_p",
        "is_berturk_all_split_subset",
        "small_subset_caveat",
        "subset_definition_caveat",
        "interpretation_caveat",
    ]
]

triplet_probe_summary_with_caveats

## Write thesis-ready tables

The table writes are additive: every output path starts with `outputs/tables/0801-`.

In [ ]:
table_outputs = {
    OUTPUT_FILES["best_overall_configurations"]: best_overall_configurations,
    OUTPUT_FILES["best_isolated_vs_contextual_configurations"]: best_isolated_vs_contextual_configurations,
    OUTPUT_FILES["clean_one_both_spearman_summary"]: fragmentation_category_spearman,
    OUTPUT_FILES["fragmentation_category_strategy_summary"]: fragmentation_category_strategy_summary,
    OUTPUT_FILES["top3_spearman_by_fragmentation_model_condition"]: top3_spearman_by_fragmentation_model_condition,
    OUTPUT_FILES["contextual_delta_by_configuration"]: contextual_delta_by_configuration,
    OUTPUT_FILES["triplet_probe_summary_with_caveats"]: triplet_probe_summary_with_caveats,
}

for path, frame in table_outputs.items():
    frame.to_csv(path, index=False, encoding="utf-8")
    print(f"Saved {len(frame):>3} rows -> {path.relative_to(PROJECT_ROOT)}")
    print(f"Preview: {path.name}")
    with pd.option_context("display.max_rows", 300, "display.max_columns", None, "display.width", 180):
        display(frame)

## Thesis-ready figures

These figures summarize the main results, fragmentation moderator pattern, contextual gain, and exploratory triplet-probe evidence.

In [ ]:
sns.set_theme(style="whitegrid", context="paper", font_scale=1.05)
plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": 300,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "font.size": 10,
    }
)

palette_input = {"isolated": "#4C78A8", "contextual": "#F58518"}
palette_model = {"BERTurk": "#4C78A8", "mBERT": "#54A24B", "XLM-R": "#F58518"}
palette_triplet = {"Same root": "#4C78A8", "Same suffix": "#E45756"}


def save_current_figure(path: Path) -> None:
    plt.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"Saved figure -> {path.relative_to(PROJECT_ROOT)}")
    if Image is not None:
        display(Image(filename=str(path)))
    else:
        print(f"Figure preview unavailable outside IPython: {path}")
    plt.close()


In [ ]:
# Figure 1: best Spearman by model and input condition.
plot_data = best_isolated_vs_contextual_configurations.copy()
plot_data = apply_display_order(plot_data)

fig, ax = plt.subplots(figsize=(7.4, 4.4))
sns.barplot(
    data=plot_data,
    x="model",
    y="spearman_rho",
    hue="input_condition",
    order=MODEL_ORDER,
    hue_order=INPUT_ORDER,
    palette=palette_input,
    ax=ax,
)
ax.set_title("Best Spearman correlation by model and input condition")
ax.set_xlabel("Model")
ax.set_ylabel("Best Spearman rho")
ax.set_ylim(0, max(0.6, plot_data["spearman_rho"].max() + 0.06))
ax.legend(title="Input condition", frameon=True, loc="upper right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
fig.tight_layout()
save_current_figure(OUTPUT_FILES["best_spearman_by_model_condition_figure"])

In [ ]:
# Figure 2: mean Spearman by fragmentation category across layer/pooling configurations.
fragmentation_plot = (
    fragmentation_category_spearman.groupby(
        ["input_condition", "model", "fragmentation_category"], observed=True
    )
    .agg(
        mean_spearman_rho=("spearman_rho", "mean"),
        sd_spearman_rho=("spearman_rho", "std"),
        n_configurations=("spearman_rho", "size"),
        n_pairs_per_configuration=("n_pairs", "first"),
    )
    .reset_index()
)
fragmentation_plot["fragmentation_label"] = fragmentation_plot["fragmentation_category"].map(
    FRAGMENTATION_LABELS
)

fig, axes = plt.subplots(1, 3, figsize=(12.2, 4.2), sharey=True)
ymin = min(-0.15, fragmentation_plot["mean_spearman_rho"].min() - 0.05)
ymax = max(0.65, fragmentation_plot["mean_spearman_rho"].max() + 0.08)
for ax, model in zip(axes, MODEL_ORDER):
    subset = fragmentation_plot[fragmentation_plot["model"].astype(str).eq(model)].copy()
    sns.barplot(
        data=subset,
        x="fragmentation_label",
        y="mean_spearman_rho",
        hue="input_condition",
        order=[FRAGMENTATION_LABELS[key] for key in FRAGMENTATION_ORDER],
        hue_order=INPUT_ORDER,
        palette=palette_input,
        ax=ax,
    )
    ax.set_title(model)
    ax.set_xlabel("")
    ax.set_ylabel("Mean Spearman rho" if ax is axes[0] else "")
    ax.set_ylim(ymin, ymax)
    ax.tick_params(axis="x", rotation=18)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=2, fontsize=7)
    if ax is not axes[-1]:
        ax.get_legend().remove()
axes[-1].legend(title="Input condition", frameon=True, loc="upper right")
fig.suptitle("Spearman correlation by fragmentation category", y=1.03, fontsize=13, fontweight="bold")
fig.tight_layout()
save_current_figure(OUTPUT_FILES["spearman_by_fragmentation_category_figure"])

In [ ]:
# Figure 3: contextual minus isolated delta by full configuration.
delta_plot = contextual_delta_by_configuration.copy()
delta_plot["configuration"] = (
    delta_plot["model"].astype(str)
    + " L"
    + delta_plot["layer"].astype(int).astype(str)
    + " "
    + delta_plot["pooling"].astype(str)
)
delta_plot = delta_plot.sort_values("contextual_minus_isolated", ascending=True)

fig, ax = plt.subplots(figsize=(8.6, 9.2))
sns.barplot(
    data=delta_plot,
    y="configuration",
    x="contextual_minus_isolated",
    hue="model",
    hue_order=MODEL_ORDER,
    dodge=False,
    palette=palette_model,
    ax=ax,
)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_title("Contextual minus isolated Spearman by configuration")
ax.set_xlabel("Contextual minus isolated Spearman rho")
ax.set_ylabel("")
ax.legend(title="Model", frameon=True, loc="lower right")
fig.tight_layout()
save_current_figure(OUTPUT_FILES["contextual_delta_by_configuration_figure"])

In [ ]:
# Figure 4: exploratory triplet same-root vs same-suffix summary.
triplet_headline = triplet_probe_summary_with_caveats.query(
    "summary_source == 'headline_all_triplets'"
).copy()
triplet_long = triplet_headline.melt(
    id_vars=["model", "mean_delta"],
    value_vars=["mean_sim_same_root", "mean_sim_same_suffix"],
    var_name="comparison",
    value_name="mean_similarity",
)
triplet_long["comparison"] = triplet_long["comparison"].map(
    {
        "mean_sim_same_root": "Same root",
        "mean_sim_same_suffix": "Same suffix",
    }
)
triplet_long["model"] = pd.Categorical(triplet_long["model"], MODEL_ORDER, ordered=True)

fig, ax = plt.subplots(figsize=(7.2, 4.5))
sns.barplot(
    data=triplet_long,
    x="model",
    y="mean_similarity",
    hue="comparison",
    order=MODEL_ORDER,
    hue_order=["Same root", "Same suffix"],
    palette=palette_triplet,
    ax=ax,
)
ax.set_title("Exploratory triplet probe: same-root vs same-suffix similarity")
ax.set_xlabel("Model")
ax.set_ylabel("Mean cosine similarity")
ax.set_ylim(0, max(1.08, triplet_long["mean_similarity"].max() + 0.1))
ax.legend(title="", frameon=True, loc="upper right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)

for idx, model in enumerate(MODEL_ORDER):
    row = triplet_headline[triplet_headline["model"].astype(str).eq(model)]
    if not row.empty:
        delta = float(row.iloc[0]["mean_delta"])
        ymax_model = triplet_long[triplet_long["model"].astype(str).eq(model)]["mean_similarity"].max()
        ax.text(idx, ymax_model + 0.055, f"Delta {delta:.3f}", ha="center", va="bottom", fontsize=8)

fig.tight_layout()
save_current_figure(OUTPUT_FILES["triplet_same_root_vs_same_suffix_figure"])

## Output verification

The final checks verify that all planned `0801-*` outputs exist and that the fixed-pair and headline-value assumptions still hold.

In [ ]:
for label, path in OUTPUT_FILES.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:7} {label:45s} {path.relative_to(PROJECT_ROOT)}")

missing_outputs = [path for path in OUTPUT_FILES.values() if not path.exists()]
if missing_outputs:
    missing_text = ", ".join(str(path.relative_to(PROJECT_ROOT)) for path in missing_outputs)
    raise FileNotFoundError(f"Missing generated outputs: {missing_text}")

if set(fragmentation_category_spearman["fragmentation_category"].astype(str)) != set(FRAGMENTATION_ORDER):
    raise AssertionError("Expected all three fragmentation categories in the Spearman summary")

if not fragmentation_category_spearman["total_pairs"].eq(500).all():
    raise AssertionError("Every configuration should retain the full 500-pair evaluation set before category splits")

best_contextual = best_overall_configurations.query("input_condition == 'contextual'").iloc[0]
if not (
    str(best_contextual["model"]) == "BERTurk"
    and int(best_contextual["layer"]) == 7
    and str(best_contextual["pooling"]) == "first"
):
    raise AssertionError("Current best contextual configuration differs from the expected snapshot")

if not math.isclose(float(best_contextual["spearman_rho"]), 0.5275, abs_tol=5e-4):
    raise AssertionError("Current best contextual Spearman value differs from the expected snapshot")

if not triplet_probe_summary_with_caveats["analysis_role"].eq("exploratory_probe").all():
    raise AssertionError("Triplet probe caveat role was not applied consistently")

print("All final-results table and figure checks passed.")